In [1]:
import pandas as pd
import numpy as np
import geo.sphere
import matplotlib.pyplot as plt

### Get route lengths

In [2]:
shapes = pd.read_csv('./output/future/shapes_rail.txt')
shapes.head()

,shape_id,shape_pt_lat,shape_pt_lon,shape_pt_sequence,shape_dist_traveled
0,L-PLR1-CLFD-WSMD,-33.781953,151.047190,1,0.000000
1,L-PLR1-CLFD-WSMD,-33.782527,151.046673,2,80.182320
2,L-PLR1-CLFD-WSMD,-33.782692,151.046507,3,104.651857
3,L-PLR1-CLFD-WSMD,-33.782793,151.046403,4,119.782310
4,L-PLR1-CLFD-WSMD,-33.782982,151.046214,5,147.692514


In [60]:
routesdf = shapes[['shape_id','shape_dist_traveled']].groupby('shape_id').max().reset_index()
routesdf

,shape_id,shape_dist_traveled
0,L-PLR1-CLFD-WSMD,11063.625622
1,L-PLR1-WSMD-CLFD,11064.789528
2,L-PLR2-OLYP-WSUN,10605.698872
3,L-PLR2-WSUN-OLYP,10603.191242
4,M-1-LVPL-TLWG,82845.453020
5,M-1-TLWG-LVPL,82916.833796
6,M-2-BKTN-EPNG,46962.993014
7,M-2-EPNG-BKTN,46980.295166
8,M-3a-MBRA-WSIA,77768.046026
9,M-3a-WSIA-MBRA,77948.560230


### Get metro, lightrail and train effective speeds from current gtfs

I need to get the length of each route (between stops) and the total travel time (between stops), both are available in stop_times.txt. I can get speed from length and time. I need to do this separately for metro, train and lightrail. 

In [2]:
trips = pd.read_csv("./input/gtfs_static_8jul2020/trips.txt")
stoptimes = pd.read_csv("./input/gtfs_static_8jul2020/stop_times.txt")
routes = pd.read_csv("./input/gtfs_static_8jul2020/routes.txt")

/Users/hemarayaprolu/anaconda3/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3063: DtypeWarning: Columns (0,3,5,10) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [3]:
routes['route_desc'].unique()

array(['Temporary buses', 'Illawarra Buses Network',
       'Intercity Trains Network', 'Sydney Trains Network',
       'School buses', 'Sydney Buses Network', 'Private bus services',
       'Sydney Metro Network', 'Private ferry and fast ferry services',
       'Hunter Buses Network', 'Private coach services',
       'Central Coast Buses Network', 'Blue Mountains Buses Network',
       'Newcastle Ferries', 'Regional Trains and Coaches Network',
       'Sydney Light Rail Network', 'Newcastle Light Rail',
       'Sydney Ferries Network'], dtype=object)

In [4]:
routes_m = routes.loc[routes['route_desc']=='Sydney Metro Network'].reset_index(drop=True)
print(len(routes_m))
routes_t = routes.loc[routes['route_desc']=='Sydney Trains Network'].reset_index(drop=True)
print(len(routes_t))
routes_l = routes.loc[routes['route_desc']=='Sydney Light Rail Network'].reset_index(drop=True)
print(len(routes_l))

1
9
4


In [5]:
routes_l

,route_id,agency_id,route_short_name,route_long_name,route_desc,route_type,route_color,route_text_color,exact_times
0,78-L1-sj2-1,LR,L1,L1 Dulwich Hill Line,Sydney Light Rail Network,0,EE343F,FFFFFF,0
1,78-L2-sj2-1,SLR,L2,L2 Randwick Line,Sydney Light Rail Network,0,EE343F,FFFFFF,0
2,78-L3-sj2-1,SLR,L3,L3 Kingsford Line,Sydney Light Rail Network,0,EE343F,FFFFFF,0
3,78-LX-sj2-1,SLR,LX,LX Special Event Services,Sydney Light Rail Network,0,EE343F,FFFFFF,1


In [6]:
# Removing speacial event service
routes_l = routes_l[:3]

In [7]:
trips['route_id'] = trips['route_id'].astype(str)
routes['route_id'] = routes['route_id'].astype(str)
trips_m = trips.loc[trips['route_id'].isin(routes_m['route_id'])].reset_index(drop=True)
print(len(trips_m))
trips_t = trips.loc[trips['route_id'].isin(routes_t['route_id'])].reset_index(drop=True)
print(len(trips_t))
trips_l = trips.loc[trips['route_id'].isin(routes_l['route_id'])].reset_index(drop=True)
print(len(trips_l))

14711
36124
2370


In [8]:
stoptimes['trip_id'] = stoptimes['trip_id'].astype(str)
trips['trip_id'] = trips['trip_id'].astype(str)
stoptimes_m = stoptimes.loc[stoptimes['trip_id'].isin(trips_m['trip_id'])].reset_index(drop=True)
print(len(stoptimes_m))
stoptimes_t = stoptimes.loc[stoptimes['trip_id'].isin(trips_t['trip_id'])].reset_index(drop=True)
print(len(stoptimes_t))
stoptimes_l = stoptimes.loc[stoptimes['trip_id'].isin(trips_l['trip_id'])].reset_index(drop=True)
print(len(stoptimes_l))

191243
609829
43171


In [9]:
stoptimes_m['stop_id'].unique()

array([2155269, 2155267, 2155265, 2153402, 2153404, 2154264, 2154262,
       2126159, 2121225, 2113351, 2113341, 2113361, 2067142, 2067143,
       2113362, 2113342, 2113352, 2121226, 2126160, 2154263, 2154265,
       2153405, 2153403, 2155266, 2155268, 2155270], dtype=object)

In [49]:
metrodf = stoptimes_m[['trip_id','arrival_time']].loc[stoptimes_m['stop_sequence']==1].reset_index(drop=True)
metrodf = metrodf.merge(stoptimes_m[['trip_id','departure_time','stop_sequence','shape_dist_traveled']].groupby('trip_id').max().reset_index(), how='left', on='trip_id')
metrodf.columns = ['trip_id','start_time','end_time','stops','shape_dist_traveled']
metrodf['start_hr'] = metrodf['start_time'].apply(lambda x: int(x.split(':')[0]))
metrodf['end_hr'] = metrodf['end_time'].apply(lambda x: int(x.split(':')[0]))
print(len(metrodf))
metrodf = metrodf.loc[(metrodf['start_hr']<24) & (metrodf['end_hr']<24)]
print(len(metrodf))
metrodf['start_time'] = pd.to_datetime(metrodf['start_time'],format = '%H:%M:%S')
metrodf['end_time'] = pd.to_datetime(metrodf['end_time'],format = '%H:%M:%S')
metrodf['travel_time_h'] = (metrodf['end_time'] - metrodf['start_time'])/np.timedelta64(1,'h')
metrodf['travel_speed_kmph'] = metrodf['shape_dist_traveled']/(1000 * metrodf['travel_time_h'])

14711
14433


In [48]:
traindf = stoptimes_t[['trip_id','arrival_time']].loc[stoptimes_t['stop_sequence']==1].reset_index(drop=True)
traindf = traindf.merge(stoptimes_t[['trip_id','departure_time','stop_sequence','shape_dist_traveled']].groupby('trip_id').max().reset_index(), how='left', on='trip_id')
traindf.columns = ['trip_id','start_time','end_time','stops','shape_dist_traveled']
traindf['start_hr'] = traindf['start_time'].apply(lambda x: int(x.split(':')[0]))
traindf['end_hr'] = traindf['end_time'].apply(lambda x: int(x.split(':')[0]))
print(len(traindf))
traindf = traindf.loc[(traindf['start_hr']<24) & (traindf['end_hr']<24)]
print(len(traindf))
traindf['start_time'] = pd.to_datetime(traindf['start_time'],format = '%H:%M:%S')
traindf['end_time'] = pd.to_datetime(traindf['end_time'],format = '%H:%M:%S')
traindf['travel_time_h'] = (traindf['end_time'] - traindf['start_time'])/np.timedelta64(1,'h')
traindf['travel_speed_kmph'] = traindf['shape_dist_traveled']/(1000 * traindf['travel_time_h'])

36124
34240


In [53]:
lightraildf = stoptimes_l[['trip_id','arrival_time']].loc[stoptimes_l['stop_sequence']==1].reset_index(drop=True)
lightraildf = lightraildf.merge(stoptimes_l[['trip_id','departure_time','stop_sequence','shape_dist_traveled']].groupby('trip_id').max().reset_index(), how='left', on='trip_id')
lightraildf.columns = ['trip_id','start_time','end_time','stops','shape_dist_traveled']
lightraildf['start_hr'] = lightraildf['start_time'].apply(lambda x: int(x.split(':')[0]))
lightraildf['end_hr'] = lightraildf['end_time'].apply(lambda x: int(x.split(':')[0]))
print(len(lightraildf))
lightraildf = lightraildf.loc[(lightraildf['start_hr']<24) & (lightraildf['end_hr']<24)]
print(len(lightraildf))
lightraildf['start_time'] = pd.to_datetime(lightraildf['start_time'],format = '%H:%M:%S')
lightraildf['end_time'] = pd.to_datetime(lightraildf['end_time'],format = '%H:%M:%S')
lightraildf['travel_time_h'] = (lightraildf['end_time'] - lightraildf['start_time'])/np.timedelta64(1,'h')
lightraildf['travel_speed_kmph'] = lightraildf['shape_dist_traveled']/(1000 * lightraildf['travel_time_h'])

2370
2179


In [46]:
metrodf['travel_speed_kmph'].describe()

count    14433.000000
mean        58.712203
std          0.013050
min         58.699100
25%         58.699100
50%         58.725200
75%         58.725200
max         58.725200
Name: travel_speed_kmph, dtype: float64

In [56]:
metrodf['travel_speed_kmph'].median()

58.7252

In [52]:
traindf['travel_speed_kmph'].describe()

count    34240.000000
mean        36.492663
std          7.807310
min         15.317459
25%         30.436748
50%         35.293944
75%         41.323961
max         60.500681
Name: travel_speed_kmph, dtype: float64

In [57]:
traindf['travel_speed_kmph'].median()

35.293944303797474

In [55]:
lightraildf['travel_speed_kmph'].describe()

count    2179.000000
mean       17.247671
std         3.526147
min        10.947675
25%        13.755584
50%        20.086550
75%        20.640662
max        21.402922
Name: travel_speed_kmph, dtype: float64

In [58]:
lightraildf['travel_speed_kmph'].median()

20.086550335570468

### Get route travel times

In [59]:
# Rounding metro, train and light rail speeds to the following:

metrospeed_kmph = 59
trainspeed_kmph = 37
lightrailspeed_kmph = 20

In [61]:
routesdf.head()

,shape_id,shape_dist_traveled
0,L-PLR1-CLFD-WSMD,11063.625622
1,L-PLR1-WSMD-CLFD,11064.789528
2,L-PLR2-OLYP-WSUN,10605.698872
3,L-PLR2-WSUN-OLYP,10603.191242
4,M-1-LVPL-TLWG,82845.453020


In [71]:
routesdf['travel_speed_kmph'] = routesdf['shape_id'].apply(lambda x: metrospeed_kmph if x.startswith('M') else (trainspeed_kmph if x.startswith('T') else lightrailspeed_kmph))
routesdf['travel_time_minutes'] = np.round(routesdf['shape_dist_traveled']*60/(1000*routesdf['travel_speed_kmph']),0)
routesdf


,shape_id,shape_dist_traveled,travel_speed_kmph,travel_time_minutes
0,L-PLR1-CLFD-WSMD,11063.625622,20,33.0
1,L-PLR1-WSMD-CLFD,11064.789528,20,33.0
2,L-PLR2-OLYP-WSUN,10605.698872,20,32.0
3,L-PLR2-WSUN-OLYP,10603.191242,20,32.0
4,M-1-LVPL-TLWG,82845.453020,59,84.0
5,M-1-TLWG-LVPL,82916.833796,59,84.0
6,M-2-BKTN-EPNG,46962.993014,59,48.0
7,M-2-EPNG-BKTN,46980.295166,59,48.0
8,M-3a-MBRA-WSIA,77768.046026,59,79.0
9,M-3a-WSIA-MBRA,77948.560230,59,79.0


In [73]:
routesdf[['shape_id','travel_time_minutes']].to_csv('traveltimes.csv', index=False)

### Get headways, first and last starts

#### Metro

Metro headways from Google Maps: 

Peak = 4min

Off-peak = 10min

Considering 4 minutes peak and 8 minutes off-peak (doubling).

In [76]:
metrodf['start_time'].min()

Timestamp('1900-01-01 04:35:00')

In [77]:
metrodf['end_time'].max()

Timestamp('1900-01-01 23:53:00')

#### Train (T2)

T2 headways from Google Maps:

Peak: 3 minutes followed by 12 minutes!

Off-peak: 15 minutes

Considering 15 minutes off-peak and 7 minutes peak (also double).

In [82]:
traindf['start_time'].min()

Timestamp('1900-01-01 03:09:01')

In [85]:
traindf['end_time'].max()

Timestamp('1900-01-01 23:59:48')

In [84]:
print(len(traindf))
traindf = traindf.merge(trips_t[['trip_id','route_id']], how='left', on='trip_id')
print(len(traindf))

34240
34240


In [87]:
t2df = traindf.loc[traindf['route_id'].str.contains('T2')]
len(t2df)

4406

In [89]:
t2df['start_time'].min()

Timestamp('1900-01-01 03:28:01')

In [91]:
t2df['end_time'].max()

Timestamp('1900-01-01 23:59:00')

#### Light rail

Light rail headways from Google Maps:

Peak: 8 minutes

Off-peak: There are ranges varying from 10-12 mintues to 15 minutes

Assuming 15 minutes off-peak and 7 minutes peak (doubling).

In [92]:
lightraildf['start_time'].min()

Timestamp('1900-01-01 04:35:00')

In [93]:
lightraildf['end_time'].max()

Timestamp('1900-01-01 23:59:45')